# Análise de Demonstrações Financeiras – Empresas Listadas na CVM (2024)

Este notebook realiza a extração, limpeza e análise de dados financeiros de empresas brasileiras listadas na Comissão de Valores Mobiliários (CVM).  

O objetivo é construir um pipeline de dados que permita calcular indicadores financeiros e gerar insights sobre a rentabilidade e eficiência das empresas.  

**Indicadores calculados:**
- ROE (Return on Equity)
- ROA (Return on Assets)
- Margem Líquida
- Classificação de porte das empresas

## 0. Configuração do Ambiente

Nesta etapa importamos as bibliotecas necessárias para manipulação de dados, visualização e conexão com o banco de dados.

In [1]:
import pandas as pd
import numpy as np
import logging
from sqlalchemy import create_engine

## 1. Carga dos Dados (Extract)

Carregamos os arquivos brutos da CVM referentes às demonstrações financeiras consolidadas de 2024.

In [2]:
def extract():
    df_dre = pd.read_csv(
        'dfp_cia_aberta_DRE_con_2024.csv',
        sep=';',
        encoding='ISO-8859-1'
    )
    df_bpa = pd.read_csv(
        'dfp_cia_aberta_BPA_con_2024.csv',
        sep=';',
        encoding='ISO-8859-1'
    )
    df_bpp = pd.read_csv(
        'dfp_cia_aberta_BPP_con_2024.csv',
        sep=';',
        encoding='ISO-8859-1'
    )
    return df_dre, df_bpa, df_bpp

## Configuração de Logging

Implementamos logs para rastrear cada etapa do pipeline, facilitando a detecção de erros.

In [3]:
def setup_logging(nome_log="pipeline_cvm"):
    formatter = logging.Formatter('%(asctime)s | %(name)s | %(levelname)s | %(message)s')
    file_handler = logging.FileHandler(f"{nome_log}.log")
    file_handler.setFormatter(formatter)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger = logging.getLogger(nome_log)
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)
    return logger

log = setup_logging()

## 2. Limpeza e Padronização (Cleaner)

Nesta etapa, tratamos os dados para garantir consistência:
- Mantemos apenas o exercício 'ÚLTIMO'
- Convertendo valores em milhares para unidade
- Padronizamos datas e colunas

In [4]:
def cleaner(df_dre, df_bpa, df_bpp):
    # DRE
    dre = df_dre[(df_dre['ORDEM_EXERC']=='ÚLTIMO') &
                 (df_dre['CD_CONTA'].isin(['3.01','3.11'])) &
                 (df_dre['ST_CONTA_FIXA']=='S')].copy()
    dre['VL_CONTA'] = np.where(dre['ESCALA_MOEDA']=='MIL', dre['VL_CONTA']*1000, dre['VL_CONTA'])
    dre['DT_REFER'] = pd.to_datetime(dre['DT_REFER'])
    dre = dre.pivot_table(
        index=['CNPJ_CIA','DENOM_CIA','DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='first'
    ).reset_index()
    dre = dre.rename(columns={'3.01':'RECEITA_VENDA','3.11':'LUCRO_LÍQUIDO'})

    # BPA
    bpa = df_bpa[(df_bpa['ORDEM_EXERC']=='ÚLTIMO') &
                 (df_bpa['CD_CONTA']=='1') &
                 (df_bpa['ST_CONTA_FIXA']=='S')].copy()
    bpa['VL_CONTA'] = np.where(bpa['ESCALA_MOEDA']=='MIL', bpa['VL_CONTA']*1000, bpa['VL_CONTA'])
    bpa['DT_REFER'] = pd.to_datetime(bpa['DT_REFER'])
    bpa = bpa.pivot_table(
        index=['CNPJ_CIA','DENOM_CIA','DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='first'
    ).reset_index()
    bpa = bpa.rename(columns={'1':'ATIVO_TOTAL'})

    # BPP
    bpp = df_bpp[(df_bpp['ORDEM_EXERC']=='ÚLTIMO') &
                 (df_bpp['CD_CONTA'].isin(['2.01','2.02'])) &
                 (df_bpp['ST_CONTA_FIXA']=='S')].copy()
    bpp['VL_CONTA'] = np.where(bpp['ESCALA_MOEDA']=='MIL', bpp['VL_CONTA']*1000, bpp['VL_CONTA'])
    bpp['DT_REFER'] = pd.to_datetime(bpp['DT_REFER'])
    bpp = bpp.pivot_table(
        index=['CNPJ_CIA','DENOM_CIA','DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='first'
    ).reset_index()
    bpp['PASSIVO_TOTAL'] = bpp['2.01'] + bpp['2.02']
    bpp = bpp[['CNPJ_CIA','DT_REFER','PASSIVO_TOTAL']]

    # Merge
    df_final = pd.merge(dre, bpa[['CNPJ_CIA','DT_REFER','ATIVO_TOTAL']], on=['CNPJ_CIA','DT_REFER'], how='inner')
    df_final = pd.merge(df_final, bpp[['CNPJ_CIA','DT_REFER','PASSIVO_TOTAL']], on=['CNPJ_CIA','DT_REFER'], how='inner')

    return df_final

## 3. Transformação e Cálculo de Indicadores (Transform)

Após a limpeza, calculamos os principais indicadores financeiros que permitem avaliar a rentabilidade e eficiência das empresas:  

- **Margem Líquida**  
- **ROA (Return on Assets)**  
- **ROE (Return on Equity)**  
- **Patrimônio Líquido**  
- **Classificação de porte das empresas**

In [5]:
def transform(df_final):
    df_result = df_final.copy()

    # Margem Líquida
    df_result['MARGEM_LÍQUIDA'] = np.where(
        df_result['RECEITA_VENDA'] <= 0,
        np.nan,
        (df_result['LUCRO_LÍQUIDO'] / df_result['RECEITA_VENDA']) * 100
    )

    # ROA
    df_result['ROA'] = np.where(
        df_result['ATIVO_TOTAL'] <= 0,
        np.nan,
        (df_result['LUCRO_LÍQUIDO'] / df_result['ATIVO_TOTAL']) * 100
    )

    # Patrimônio Líquido
    df_result['PATRIMÔNIO_LÍQUIDO'] = df_result['ATIVO_TOTAL'] - df_result['PASSIVO_TOTAL']

    # ROE
    df_result['ROE'] = np.where(
        df_result['PATRIMÔNIO_LÍQUIDO'] <= 0,
        np.nan,
        (df_result['LUCRO_LÍQUIDO'] / df_result['PATRIMÔNIO_LÍQUIDO']) * 100
    )

    # Classificação de porte
    faixas = [0, 1e8, 1e9, np.inf]
    nomes = ['Pequena', 'Média', 'Grande']
    df_result['PORTE_EMPRESA'] = pd.cut(df_result['ATIVO_TOTAL'], bins=faixas, labels=nomes)

    # Excluir empresas em recuperação judicial
    df_result = df_result[~df_result['DENOM_CIA'].str.contains('RECUPERAÇÃO', case=False, na=False)]

    # Substituir infinitos por NaN
    df_result = df_result.replace([np.inf, -np.inf], np.nan)

    return df_result

## 4. Persistência em Banco de Dados (Load)

Utilizamos **SQLAlchemy** para criar um banco SQLite, permitindo consultas rápidas e estruturadas.

In [6]:
def load(df, nome_tabela='empresas_cvm'):
    try:
        engine = create_engine('sqlite:///data_cvm.db', echo=False)
        df.to_sql(
            nome_tabela,
            engine,
            if_exists='replace',
            index=False,
            chunksize=500
        )
        print(f"Sucesso! Tabela '{nome_tabela}' gerada com {len(df)} registros.")
    except Exception as e:
        print(f"Erro ao salvar no banco: {e}")

## 5. Execução do Pipeline ETL (Main)

Integramos todas as etapas: **Extract → Cleaner → Transform → Load**.

In [7]:
def main():
    log.info("--- INICIANDO ETL ---")
    try:
        log.info("Extraindo dados da CVM (Extract)...")
        df_dre, df_bpa, df_bpp = extract()

        log.info("Iniciando limpeza dos dados (Cleaner)...")
        df_limpo = cleaner(df_dre, df_bpa, df_bpp)

        log.info("Calculando indicadores financeiros (Transform)...")
        df_final = transform(df_limpo)

        log.info("Persistindo dados no banco SQL (Load)...")
        load(df_final)

        log.info("--- ETL CONCLUÍDA COM SUCESSO ---")

    except Exception as e:
        log.critical(f"FALHA NO SISTEMA: {str(e)}", exc_info=True)

In [8]:
if __name__ == "__main__":
    main()

2026-03-10 10:56:52,986 | pipeline_cvm | INFO | --- INICIANDO ETL ---
INFO:pipeline_cvm:--- INICIANDO ETL ---
2026-03-10 10:56:52,989 | pipeline_cvm | INFO | Extraindo dados da CVM (Extract)...
INFO:pipeline_cvm:Extraindo dados da CVM (Extract)...
2026-03-10 10:56:53,699 | pipeline_cvm | INFO | Iniciando limpeza dos dados (Cleaner)...
INFO:pipeline_cvm:Iniciando limpeza dos dados (Cleaner)...
2026-03-10 10:56:53,816 | pipeline_cvm | INFO | Calculando indicadores financeiros (Transform)...
INFO:pipeline_cvm:Calculando indicadores financeiros (Transform)...
2026-03-10 10:56:53,830 | pipeline_cvm | INFO | Persistindo dados no banco SQL (Load)...
INFO:pipeline_cvm:Persistindo dados no banco SQL (Load)...
2026-03-10 10:56:53,911 | pipeline_cvm | INFO | --- ETL CONCLUÍDA COM SUCESSO ---
INFO:pipeline_cvm:--- ETL CONCLUÍDA COM SUCESSO ---


Sucesso! Tabela 'empresas_cvm' gerada com 449 registros.


### Diferenciais do Pipeline

- **Logging:** Cada etapa do ETL é registrada, facilitando debug em produção.  
- **Tratamento de Exceções:** Blocos `try-except` garantem que erros não travem todo o pipeline.  
- **Modularização:** Funções separadas para Cleaner, Transform e Load, seguindo boas práticas de engenharia de software.